Q1. Customer Satisfaction and Ordinal Encoding

(a) Apply OrdinalEncoder to Satisfaction and Membership

In [1]:

import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

df = pd.DataFrame({
    "Customer_ID": range(101, 113),
    "Satisfaction": [
        "Poor", "Good", "Average", "Excellent",
        "Poor", "Good", "Average", "Excellent",
        "Good", "Average", "Excellent", "Poor"
    ],
    "Purchase_Amount": [
        500, 1200, 800, 2500,
        400, 1500, 900, 3000,
        1800, 700, 2800, 350
    ],
    "Membership": [
        "Bronze", "Silver", "Bronze", "Gold",
        "Bronze", "Gold", "Silver", "Gold",
        "Silver", "Bronze", "Gold", "Silver"
    ]
})

# Define category order
satisfaction_order = [
    "Poor", "Average", "Good", "Excellent"
]

membership_order = [
    "Bronze", "Silver", "Gold"
]

# Create Ordinal Encoder
encoder = OrdinalEncoder(
    categories=[satisfaction_order, membership_order]
)

# Apply encoding
df[["Satisfaction_Encoded", "Membership_Encoded"]] = (
    encoder.fit_transform(
        df[["Satisfaction", "Membership"]]
    )
)

print(df)

# Verify rankings
satisfaction_mapping = dict(
    zip(satisfaction_order, range(4))
)

membership_mapping = dict(
    zip(membership_order, range(3))
)

print("\nSatisfaction Ranking:", satisfaction_mapping)
print("Membership Ranking:", membership_mapping)

assert list(satisfaction_mapping.values()) == [0, 1, 2, 3]
assert list(membership_mapping.values()) == [0, 1, 2]

print("\nRanking verified successfully!")


    Customer_ID Satisfaction  Purchase_Amount Membership  \
0           101         Poor              500     Bronze   
1           102         Good             1200     Silver   
2           103      Average              800     Bronze   
3           104    Excellent             2500       Gold   
4           105         Poor              400     Bronze   
5           106         Good             1500       Gold   
6           107      Average              900     Silver   
7           108    Excellent             3000       Gold   
8           109         Good             1800     Silver   
9           110      Average              700     Bronze   
10          111    Excellent             2800       Gold   
11          112         Poor              350     Silver   

    Satisfaction_Encoded  Membership_Encoded  
0                    0.0                 0.0  
1                    2.0                 1.0  
2                    1.0                 0.0  
3                    3.0       

(b) Calculate average purchase amount before and after encoding

In [2]:

# Average before encoding
before_encoding = df.groupby(
    "Satisfaction"
)["Purchase_Amount"].mean()

print("Average Before Encoding:")
print(before_encoding)

# Average after encoding
after_encoding = df.groupby(
    "Satisfaction_Encoded"
)["Purchase_Amount"].mean()

print("\nAverage After Encoding:")
print(after_encoding)


Average Before Encoding:
Satisfaction
Average       800.000000
Excellent    2766.666667
Good         1500.000000
Poor          416.666667
Name: Purchase_Amount, dtype: float64

Average After Encoding:
Satisfaction_Encoded
0.0     416.666667
1.0     800.000000
2.0    1500.000000
3.0    2766.666667
Name: Purchase_Amount, dtype: float64


Q2. Label Encoding vs. Ordinal Encoding in Classification

(a) Apply Ordinal Encoding and Label Encoding

In [3]:

import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

df = pd.DataFrame({
    "Study_Level": [
        "Low", "Medium", "High", "Low", "High",
        "Medium", "High", "Low", "Medium", "High",
        "Low", "Medium"
    ],
    "City": [
        "Bhopal", "Delhi", "Mumbai", "Delhi",
        "Bhopal", "Mumbai", "Delhi", "Mumbai",
        "Bhopal", "Delhi", "Mumbai", "Bhopal"
    ],
    "Result": [
        "Fail", "Pass", "Pass", "Fail",
        "Pass", "Pass", "Pass", "Fail",
        "Pass", "Pass", "Fail", "Pass"
    ]
})

# Ordinal Encoding
oe = OrdinalEncoder(
    categories=[["Low", "Medium", "High"]]
)

df["Study_Level_Encoded"] = oe.fit_transform(
    df[["Study_Level"]]
).ravel()

# Label Encoding for target
le = LabelEncoder()

df["Result_Encoded"] = le.fit_transform(
    df["Result"]
)

# Label Encoding for City (demonstration only)
city_encoder = LabelEncoder()

df["City_Label"] = city_encoder.fit_transform(
    df["City"]
)

print(df)

print("\nCity Mapping:")
print(dict(zip(
    city_encoder.classes_,
    city_encoder.transform(city_encoder.classes_)
)))

print("\nResult Mapping:")
print(dict(zip(
    le.classes_,
    le.transform(le.classes_)
)))


   Study_Level    City Result  Study_Level_Encoded  Result_Encoded  City_Label
0          Low  Bhopal   Fail                  0.0               0           0
1       Medium   Delhi   Pass                  1.0               1           1
2         High  Mumbai   Pass                  2.0               1           2
3          Low   Delhi   Fail                  0.0               0           1
4         High  Bhopal   Pass                  2.0               1           0
5       Medium  Mumbai   Pass                  1.0               1           2
6         High   Delhi   Pass                  2.0               1           1
7          Low  Mumbai   Fail                  0.0               0           2
8       Medium  Bhopal   Pass                  1.0               1           0
9         High   Delhi   Pass                  2.0               1           1
10         Low  Mumbai   Fail                  0.0               0           2
11      Medium  Bhopal   Pass                  1.0  

(b) Build preprocessing workflow and train classification model

In [4]:

import pandas as pd

from sklearn.preprocessing import (
    OrdinalEncoder,
    OneHotEncoder,
    LabelEncoder
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Features and target
X = df[["Study_Level", "City"]]
y = df["Result"]

# Encode target
le = LabelEncoder()
y = le.fit_transform(y)

# Preprocessing
preprocessor = ColumnTransformer([
    ("ordinal",
     OrdinalEncoder(
         categories=[["Low", "Medium", "High"]]
     ),
     ["Study_Level"]),

    ("onehot",
     OneHotEncoder(handle_unknown="ignore"),
     ["City"])
])

# Create pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression())
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Train model
model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

print("Actual Values:", y_test)
print("Predicted Values:", y_pred)

# Evaluation
print("\nAccuracy:",
      accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=le.classes_,
    zero_division=0
))


Actual Values: [0 1 1]
Predicted Values: [0 1 1]

Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

        Fail       1.00      1.00      1.00         1
        Pass       1.00      1.00      1.00         2

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



Q3. Encoding Strategy for Mixed Categorical Data 

(a) Identify ordinal and nominal columns and apply OrdinalEncoder

In [5]:

import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

df = pd.DataFrame({
    "Size": [
        "Small", "Medium", "Large", "Medium",
        "Small", "Large", "XL", "Medium",
        "XL", "Small", "Large", "XL"
    ],
    "Color": [
        "Red", "Blue", "Green", "Red",
        "Green", "Blue", "Red", "Green",
        "Blue", "Red", "Blue", "Green"
    ],
    "Satisfaction": [
        "Poor", "Good", "Average", "Excellent",
        "Average", "Good", "Excellent", "Poor",
        "Good", "Average", "Excellent", "Good"
    ],
    "Order_Value": [
        500, 900, 1500, 1100,
        650, 1700, 2200, 700,
        2100, 550, 1600, 2300
    ]
})

# Define category orders
size_order = ["Small", "Medium", "Large", "XL"]

satisfaction_order = [
    "Poor", "Average", "Good", "Excellent"
]

# Apply ordinal encoding
encoder = OrdinalEncoder(
    categories=[size_order, satisfaction_order]
)

df[["Size_Encoded", "Satisfaction_Encoded"]] = (
    encoder.fit_transform(
        df[["Size", "Satisfaction"]]
    )
)

print(df)

# Verify mappings
print("\nSize Mapping:")
print(dict(zip(size_order, range(4))))

print("\nSatisfaction Mapping:")
print(dict(zip(satisfaction_order, range(4))))

# Check nominal feature
print("\nUnique Colors:")
print(df["Color"].unique())

# Verify encoded values
assert df["Size_Encoded"].between(0, 3).all()
assert df["Satisfaction_Encoded"].between(0, 3).all()

print("\nEncoding verified successfully!")


      Size  Color Satisfaction  Order_Value  Size_Encoded  \
0    Small    Red         Poor          500           0.0   
1   Medium   Blue         Good          900           1.0   
2    Large  Green      Average         1500           2.0   
3   Medium    Red    Excellent         1100           1.0   
4    Small  Green      Average          650           0.0   
5    Large   Blue         Good         1700           2.0   
6       XL    Red    Excellent         2200           3.0   
7   Medium  Green         Poor          700           1.0   
8       XL   Blue         Good         2100           3.0   
9    Small    Red      Average          550           0.0   
10   Large   Blue    Excellent         1600           2.0   
11      XL  Green         Good         2300           3.0   

    Satisfaction_Encoded  
0                    0.0  
1                    2.0  
2                    1.0  
3                    3.0  
4                    1.0  
5                    2.0  
6                

(b) Apply ColumnTransformer using OrdinalEncoder and OneHotEncoder

In [6]:

import pandas as pd

from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Select categorical features
X = df[["Size", "Satisfaction", "Color"]]

# Create ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("ordinal",
         OrdinalEncoder(
             categories=[
                 ["Small", "Medium", "Large", "XL"],
                 ["Poor", "Average", "Good", "Excellent"]
             ]
         ),
         ["Size", "Satisfaction"]),

        ("nominal",
         OneHotEncoder(handle_unknown="ignore"),
         ["Color"])
    ]
)

# Transform data
encoded_data = preprocessor.fit_transform(X)

# Get column names
columns = preprocessor.get_feature_names_out()

# Convert into DataFrame
encoded_df = pd.DataFrame(
    encoded_data.toarray()
    if hasattr(encoded_data, "toarray")
    else encoded_data,
    columns=columns
)

print("Encoded Dataset:")
print(encoded_df)

print("\nEncoded Column Names:")
print(columns)


Encoded Dataset:
    ordinal__Size  ordinal__Satisfaction  nominal__Color_Blue  \
0             0.0                    0.0                  0.0   
1             1.0                    2.0                  1.0   
2             2.0                    1.0                  0.0   
3             1.0                    3.0                  0.0   
4             0.0                    1.0                  0.0   
5             2.0                    2.0                  1.0   
6             3.0                    3.0                  0.0   
7             1.0                    0.0                  0.0   
8             3.0                    2.0                  1.0   
9             0.0                    1.0                  0.0   
10            2.0                    3.0                  1.0   
11            3.0                    2.0                  0.0   

    nominal__Color_Green  nominal__Color_Red  
0                    0.0                 1.0  
1                    0.0                 0.